# Price Optimization

Generate next-week price recommendations by maximizing predicted revenue.

In [85]:
import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "Data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"


In [86]:
data = pd.read_csv(PROCESSED_DIR / "data.csv")  # Load your data into a DataFrame

In [87]:
last_week = data['week'].max()
last_week

np.int64(145)

In [88]:
recent_data = data[data['week'] >= last_week - 3]
recent_data.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,...,category_Other Snacks,category_Pasta,category_Pizza,category_Rice Bowl,category_Salad,category_Sandwich,category_Seafood,category_Soup,category_Starters,revenue
443435,142,55,1885,148.47,148.47,0,0,121,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17964.87
443436,142,55,1993,152.35,152.35,0,0,189,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28794.15
443437,142,55,2539,147.44,147.44,0,0,82,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12090.08
443438,142,55,2139,292.03,292.03,0,0,14,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4088.42
443439,142,55,2631,148.44,165.93,0,0,14,647,56,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2078.16


In [89]:
recent_weekly_meals = (
    recent_data
    .groupby(['meal_id', 'week'], as_index=False)
    .agg(
        weekly_orders=('num_orders', 'sum'),
        weekly_price=('checkout_price', 'mean')
    )
)
recent_weekly_meals.head()

,meal_id,week,weekly_orders,weekly_price
0,1062,142,34990,172.171948
1,1062,143,37333,172.362078
2,1062,144,35981,172.048701
3,1062,145,36144,172.453766
4,1109,142,35948,292.909740


In [90]:
baseline = (
    recent_weekly_meals
    .groupby('meal_id', as_index=False)
    .agg(
        current_price=('weekly_price', 'median'),
        current_orders=('weekly_orders', 'mean')
    )
)
baseline.head()

,meal_id,current_price,current_orders
0,1062,172.267013,36112.00
1,1109,294.432078,41836.25
2,1198,182.973055,6751.25
3,1207,368.877697,8332.25
4,1216,353.559605,6221.75


In [91]:
# Step 2: Historical price limits using quantiles

price_bounds = (
    data
    .groupby('meal_id', as_index=False)
    .agg(
        historical_min_price=('checkout_price', lambda x: x.quantile(0.05)),
        historical_max_price=('checkout_price', lambda x: x.quantile(0.95))
    )
)

price_bounds.head()

,meal_id,historical_min_price,historical_max_price
0,1062,154.2300,191.096
1,1109,193.0495,320.130
2,1198,132.8900,228.010
3,1207,287.1500,466.630
4,1216,292.0300,445.230


In [92]:
optimization_data = baseline.merge(
    price_bounds,
    on='meal_id',
    how='left'
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price
0,1062,172.267013,36112.00,154.2300,191.096
1,1109,294.432078,41836.25,193.0495,320.130
2,1198,182.973055,6751.25,132.8900,228.010
3,1207,368.877697,8332.25,287.1500,466.630
4,1216,353.559605,6221.75,292.0300,445.230


In [93]:
avg_elasticity_per_meal = pd.read_csv(PROCESSED_DIR / "avg_elasticity_per_meal.csv")
avg_elasticity_per_meal.head()

,meal_id,avg_elasticity,median_elasticity,observations
0,1062,-34.560752,-1.494312,144
1,1109,-8.724349,-1.298453,144
2,1198,-4.297801,-0.238326,144
3,1207,-37.672884,-1.421276,144
4,1216,464.082367,-1.267532,143


In [94]:
avg_elasticity_per_meal.columns

Index(['meal_id', 'avg_elasticity', 'median_elasticity', 'observations'], dtype='object')

In [95]:
elasticity_table = avg_elasticity_per_meal[
    ['meal_id', 'median_elasticity', 'observations']
]

elasticity_table.head()

,meal_id,median_elasticity,observations
0,1062,-1.494312,144
1,1109,-1.298453,144
2,1198,-0.238326,144
3,1207,-1.421276,144
4,1216,-1.267532,143


In [96]:
optimization_data = optimization_data.merge(
    elasticity_table,
    on='meal_id',
    how='left'
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143


In [97]:
# First, remove old elasticity columns if they already exist
optimization_data = optimization_data.drop(
    columns=[
        'avg_elasticity',
        'median_elasticity',
        'observations',
        'avg_elasticity_x',
        'avg_elasticity_y',
        'median_elasticity_x',
        'median_elasticity_y',
        'observations_x',
        'observations_y'
    ],
    errors='ignore'
)

# Then merge elasticity cleanly
optimization_data = optimization_data.merge(
    elasticity_table,
    on='meal_id',
    how='left'
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143


In [98]:
max_change = 0.2

optimization_data['change_min_price'] = (
    optimization_data['current_price'] * (1 - max_change)
)

optimization_data['change_max_price'] = (
    optimization_data['current_price'] * (1 + max_change)
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations,change_min_price,change_max_price
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144,137.813610,206.720416
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144,235.545662,353.318494
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144,146.378444,219.567666
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144,295.102158,442.653237
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143,282.847684,424.271526


In [99]:
optimization_data['min_allowed_price'] = optimization_data[
    ['historical_min_price', 'change_min_price']
].max(axis=1)

optimization_data['max_allowed_price'] = optimization_data[
    ['historical_max_price', 'change_max_price']
].min(axis=1)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations,change_min_price,change_max_price,min_allowed_price,max_allowed_price
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144,137.813610,206.720416,154.230000,191.096000
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144,235.545662,353.318494,235.545662,320.130000
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144,146.378444,219.567666,146.378444,219.567666
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144,295.102158,442.653237,295.102158,442.653237
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143,282.847684,424.271526,292.030000,424.271526


In [100]:
broken_ranges = optimization_data[
    optimization_data['min_allowed_price'] >= optimization_data['max_allowed_price']
]

broken_ranges.shape

(0, 11)

In [101]:
optimization_data['possible_prices'] = optimization_data.apply(
    lambda row: np.linspace(
        row['min_allowed_price'],
        row['max_allowed_price'],
        20
    ),
    axis=1
)

optimization_data[['meal_id', 'current_price', 'min_allowed_price', 'max_allowed_price', 'possible_prices']].head()

,meal_id,current_price,min_allowed_price,max_allowed_price,possible_prices
0,1062,172.267013,154.230000,191.096000,"[154.23, 156.17031578947368, 158.1106315789473..."
1,1109,294.432078,235.545662,320.130000,"[235.54566233766238, 239.99746958304857, 244.4..."
2,1198,182.973055,146.378444,219.567666,"[146.37844396551722, 150.23050828039925, 154.0..."
3,1207,368.877697,295.102158,442.653237,"[295.1021578947369, 302.8680041551247, 310.633..."
4,1216,353.559605,292.030000,424.271526,"[292.03, 298.99008033240995, 305.9501606648199..."


In [102]:
def predict_orders_for_prices(row):
    current_price = row['current_price']
    current_orders = row['current_orders']
    elasticity = row['median_elasticity']
    possible_prices = row['possible_prices']
    
    predicted_orders = current_orders * (
        possible_prices / current_price
    ) ** elasticity
    
    return predicted_orders

In [103]:
print(optimization_data.columns)
print(avg_elasticity_per_meal.columns)

Index(['meal_id', 'current_price', 'current_orders', 'historical_min_price',
       'historical_max_price', 'median_elasticity', 'observations',
       'change_min_price', 'change_max_price', 'min_allowed_price',
       'max_allowed_price', 'possible_prices'],
      dtype='object')
Index(['meal_id', 'avg_elasticity', 'median_elasticity', 'observations'], dtype='object')


In [104]:
optimization_data['predicted_orders_list'] = optimization_data.apply(
    predict_orders_for_prices,
    axis=1
)

optimization_data[
    ['meal_id', 'possible_prices', 'predicted_orders_list']
].head()

,meal_id,possible_prices,predicted_orders_list
0,1062,"[154.23, 156.17031578947368, 158.1106315789473...","[42601.82609554212, 41813.32030402523, 41048.8..."
1,1109,"[235.54566233766238, 239.99746958304857, 244.4...","[55896.654909900724, 54554.098460334506, 53267..."
2,1198,"[146.37844396551722, 150.23050828039925, 154.0...","[7120.005459783575, 7076.0643404769435, 7033.4..."
3,1207,"[295.1021578947369, 302.8680041551247, 310.633...","[11441.899697021723, 11027.186267574698, 10637..."
4,1216,"[292.03, 298.99008033240995, 305.9501606648199...","[7927.975073063871, 7694.781614586194, 7473.58..."


In [105]:
def calculate_revenue_for_prices(row):
    possible_prices = row['possible_prices']
    predicted_orders = row['predicted_orders_list']
    
    expected_revenues = possible_prices * predicted_orders
    
    return expected_revenues

In [106]:
optimization_data['expected_revenues_list'] = optimization_data.apply(
    calculate_revenue_for_prices,
    axis=1
)

In [107]:
optimization_data[
    ['meal_id', 'possible_prices', 'predicted_orders_list', 'expected_revenues_list']
].head()

,meal_id,possible_prices,predicted_orders_list,expected_revenues_list
0,1062,"[154.23, 156.17031578947368, 158.1106315789473...","[42601.82609554212, 41813.32030402523, 41048.8...","[6570479.638715461, 6529999.436086031, 6490263..."
1,1109,"[235.54566233766238, 239.99746958304857, 244.4...","[55896.654909900724, 54554.098460334506, 53267...","[13166214.603212314, 13092845.585864767, 13021..."
2,1198,"[146.37844396551722, 150.23050828039925, 154.0...","[7120.005459783575, 7076.0643404769435, 7033.4...","[1042215.3202291067, 1063040.7424946593, 10837..."
3,1207,"[295.1021578947369, 302.8680041551247, 310.633...","[11441.899697021723, 11027.186267574698, 10637...","[3376529.2910062466, 3339781.8963071476, 33043..."
4,1216,"[292.03, 298.99008033240995, 305.9501606648199...","[7927.975073063871, 7694.781614586194, 7473.58...","[2315206.560586842, 2300663.3730854774, 228654..."


In [109]:
def choose_best_price(row):
    possible_prices = row['possible_prices']
    predicted_orders = row['predicted_orders_list']
    expected_revenues = row['expected_revenues_list']
    
    best_index = np.argmax(expected_revenues)
    
    best_price = possible_prices[best_index]
    best_predicted_orders = predicted_orders[best_index]
    best_expected_revenue = expected_revenues[best_index]
    
    return pd.Series({
        'recommended_price': best_price,
        'predicted_orders_at_recommended_price': best_predicted_orders,
        'expected_revenue_at_recommended_price': best_expected_revenue
    })

In [110]:
best_price_results = optimization_data.apply(
    choose_best_price,
    axis=1
)

best_price_results.head()

,recommended_price,predicted_orders_at_recommended_price,expected_revenue_at_recommended_price
0,154.230000,42601.826096,6.570480e+06
1,235.545662,55896.654910,1.316621e+07
2,219.567666,6464.177494,1.419324e+06
3,295.102158,11441.899697,3.376529e+06
4,292.030000,7927.975073,2.315207e+06


In [111]:
optimization_data = pd.concat(
    [optimization_data, best_price_results],
    axis=1
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations,change_min_price,change_max_price,min_allowed_price,max_allowed_price,possible_prices,predicted_orders_list,expected_revenues_list,recommended_price,predicted_orders_at_recommended_price,expected_revenue_at_recommended_price
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144,137.813610,206.720416,154.230000,191.096000,"[154.23, 156.17031578947368, 158.1106315789473...","[42601.82609554212, 41813.32030402523, 41048.8...","[6570479.638715461, 6529999.436086031, 6490263...",154.230000,42601.826096,6.570480e+06
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144,235.545662,353.318494,235.545662,320.130000,"[235.54566233766238, 239.99746958304857, 244.4...","[55896.654909900724, 54554.098460334506, 53267...","[13166214.603212314, 13092845.585864767, 13021...",235.545662,55896.654910,1.316621e+07
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144,146.378444,219.567666,146.378444,219.567666,"[146.37844396551722, 150.23050828039925, 154.0...","[7120.005459783575, 7076.0643404769435, 7033.4...","[1042215.3202291067, 1063040.7424946593, 10837...",219.567666,6464.177494,1.419324e+06
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144,295.102158,442.653237,295.102158,442.653237,"[295.1021578947369, 302.8680041551247, 310.633...","[11441.899697021723, 11027.186267574698, 10637...","[3376529.2910062466, 3339781.8963071476, 33043...",295.102158,11441.899697,3.376529e+06
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143,282.847684,424.271526,292.030000,424.271526,"[292.03, 298.99008033240995, 305.9501606648199...","[7927.975073063871, 7694.781614586194, 7473.58...","[2315206.560586842, 2300663.3730854774, 228654...",292.030000,7927.975073,2.315207e+06


In [112]:
optimization_data = pd.concat(
    [optimization_data, best_price_results],
    axis=1
)

optimization_data.head()

,meal_id,current_price,current_orders,historical_min_price,historical_max_price,median_elasticity,observations,change_min_price,change_max_price,min_allowed_price,max_allowed_price,possible_prices,predicted_orders_list,expected_revenues_list,recommended_price,predicted_orders_at_recommended_price,expected_revenue_at_recommended_price,recommended_price,predicted_orders_at_recommended_price,expected_revenue_at_recommended_price
0,1062,172.267013,36112.00,154.2300,191.096,-1.494312,144,137.813610,206.720416,154.230000,191.096000,"[154.23, 156.17031578947368, 158.1106315789473...","[42601.82609554212, 41813.32030402523, 41048.8...","[6570479.638715461, 6529999.436086031, 6490263...",154.230000,42601.826096,6.570480e+06,154.230000,42601.826096,6.570480e+06
1,1109,294.432078,41836.25,193.0495,320.130,-1.298453,144,235.545662,353.318494,235.545662,320.130000,"[235.54566233766238, 239.99746958304857, 244.4...","[55896.654909900724, 54554.098460334506, 53267...","[13166214.603212314, 13092845.585864767, 13021...",235.545662,55896.654910,1.316621e+07,235.545662,55896.654910,1.316621e+07
2,1198,182.973055,6751.25,132.8900,228.010,-0.238326,144,146.378444,219.567666,146.378444,219.567666,"[146.37844396551722, 150.23050828039925, 154.0...","[7120.005459783575, 7076.0643404769435, 7033.4...","[1042215.3202291067, 1063040.7424946593, 10837...",219.567666,6464.177494,1.419324e+06,219.567666,6464.177494,1.419324e+06
3,1207,368.877697,8332.25,287.1500,466.630,-1.421276,144,295.102158,442.653237,295.102158,442.653237,"[295.1021578947369, 302.8680041551247, 310.633...","[11441.899697021723, 11027.186267574698, 10637...","[3376529.2910062466, 3339781.8963071476, 33043...",295.102158,11441.899697,3.376529e+06,295.102158,11441.899697,3.376529e+06
4,1216,353.559605,6221.75,292.0300,445.230,-1.267532,143,282.847684,424.271526,292.030000,424.271526,"[292.03, 298.99008033240995, 305.9501606648199...","[7927.975073063871, 7694.781614586194, 7473.58...","[2315206.560586842, 2300663.3730854774, 228654...",292.030000,7927.975073,2.315207e+06,292.030000,7927.975073,2.315207e+06


In [113]:
# Show duplicated columns
optimization_data.columns[optimization_data.columns.duplicated()].tolist()

['recommended_price',
 'predicted_orders_at_recommended_price',
 'expected_revenue_at_recommended_price']

In [115]:
# 1. Remove duplicate columns by keeping only the first version
optimization_data = optimization_data.loc[:, ~optimization_data.columns.duplicated()].copy()

# 2. Remove old result columns completely
result_columns = [
    'recommended_price',
    'predicted_orders_at_recommended_price',
    'expected_revenue_at_recommended_price',
    'current_revenue',
    'revenue_uplift',
    'revenue_uplift_pct'
]

optimization_data = optimization_data.drop(
    columns=result_columns,
    errors='ignore'
)

# 3. Check that duplicate columns are gone
optimization_data.columns[optimization_data.columns.duplicated()].tolist()

[]

In [116]:
def choose_best_price(row):
    possible_prices = row['possible_prices']
    predicted_orders = row['predicted_orders_list']
    expected_revenues = row['expected_revenues_list']
    
    best_index = np.argmax(expected_revenues)
    
    best_price = possible_prices[best_index]
    best_predicted_orders = predicted_orders[best_index]
    best_expected_revenue = expected_revenues[best_index]
    
    return pd.Series({
        'recommended_price': best_price,
        'predicted_orders_at_recommended_price': best_predicted_orders,
        'expected_revenue_at_recommended_price': best_expected_revenue
    })


best_price_results = optimization_data.apply(
    choose_best_price,
    axis=1
)

optimization_data = pd.concat(
    [optimization_data, best_price_results],
    axis=1
)

optimization_data[
    [
        'meal_id',
        'current_price',
        'recommended_price',
        'current_orders',
        'predicted_orders_at_recommended_price',
        'expected_revenue_at_recommended_price'
    ]
].head()

,meal_id,current_price,recommended_price,current_orders,predicted_orders_at_recommended_price,expected_revenue_at_recommended_price
0,1062,172.267013,154.230000,36112.00,42601.826096,6.570480e+06
1,1109,294.432078,235.545662,41836.25,55896.654910,1.316621e+07
2,1198,182.973055,219.567666,6751.25,6464.177494,1.419324e+06
3,1207,368.877697,295.102158,8332.25,11441.899697,3.376529e+06
4,1216,353.559605,292.030000,6221.75,7927.975073,2.315207e+06


In [117]:
optimization_data['current_revenue'] = (
    optimization_data['current_price'] * optimization_data['current_orders']
)

optimization_data['revenue_uplift'] = (
    optimization_data['expected_revenue_at_recommended_price']
    - optimization_data['current_revenue']
)

optimization_data['revenue_uplift_pct'] = (
    optimization_data['revenue_uplift'] / optimization_data['current_revenue']
) * 100

In [118]:
optimization_data = optimization_data.sort_values(
    'revenue_uplift',
    ascending=False
)

optimization_data[
    [
        'meal_id',
        'current_price',
        'recommended_price',
        'current_orders',
        'predicted_orders_at_recommended_price',
        'current_revenue',
        'expected_revenue_at_recommended_price',
        'revenue_uplift',
        'revenue_uplift_pct'
    ]
].head(20)

,meal_id,current_price,recommended_price,current_orders,predicted_orders_at_recommended_price,current_revenue,expected_revenue_at_recommended_price,revenue_uplift,revenue_uplift_pct
24,1962,536.745714,438.500000,26961.25,140528.706358,1.447134e+07,6.162184e+07,4.715050e+07,325.819982
41,2581,537.094207,438.440000,18619.50,107862.836494,1.000043e+07,4.729138e+07,3.729096e+07,372.893694
35,2490,283.050130,226.440104,16820.50,144238.772854,4.761045e+06,3.266144e+07,2.790040e+07,586.014199
48,2826,350.604935,280.483948,29914.75,81019.525712,1.048826e+07,2.272468e+07,1.223642e+07,116.667766
13,1558,536.708506,437.530000,15042.75,31647.922744,8.073572e+06,1.384692e+07,5.773344e+06,71.509164
39,2569,276.443182,331.731818,33706.25,45040.363981,9.317863e+06,1.494132e+07,5.623459e+06,60.351379
46,2707,225.892208,182.390000,38847.00,70851.111412,8.775235e+06,1.292253e+07,4.147300e+06,47.261410
16,1754,311.720260,249.376208,42094.00,67405.128135,1.312155e+07,1.680924e+07,3.687683e+06,28.104011
15,1727,442.364156,353.891325,23777.00,40049.656340,1.051809e+07,1.417323e+07,3.655133e+06,34.750915
14,1571,587.160808,469.728646,5138.25,13435.529230,3.016979e+06,6.311053e+06,3.294074e+06,109.184516
